In [5]:
import pandas as pd
import re
import pathlib

In [15]:
data_path = pathlib.Path(".")
data = pd.read_csv(data_path / "../data_dump/listings_with_scores_lat_long.csv")
data = data.drop(columns=["spaciousness_bathroom", "spaciousness_bedroom", "spaciousness_kitchen","spaciousness_living_room","spaciousness_toilet"])
data.head()

,level_0,source_id,url,price_man_yen,area_sqm,year_built,floor_number,floors_total,address,nearest_station,...,brightness_kitchen,brightness_living_room,brightness_toilet,condition_bathroom,condition_bedroom,condition_kitchen,condition_living_room,condition_toilet,latitude,longitude
0,1,20277732,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...,2390.0,52.89,1989,4.0,4.0,東京都足立区六月１,竹ノ塚駅,...,0.7750,NaN,0.55,0.892500,0.85,0.8750,NaN,0.85,35.786911,139.800232
1,2,20332918,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...,1580.0,29.16,1976,6.0,9.0,東京都足立区東和５,北綾瀬駅,...,0.7875,NaN,0.75,0.950000,0.95,0.8875,NaN,0.85,35.775234,139.838226
2,3,20344616,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...,2380.0,53.04,1974,3.0,8.0,東京都足立区梅田７-22-9,梅島駅,...,0.7500,NaN,0.75,0.866667,0.93,0.9500,NaN,0.95,35.769596,139.801437
3,4,78385243,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_7...,1700.0,40.10,1985,1.0,5.0,東京都足立区青井１-16-9,青井駅,...,NaN,NaN,NaN,0.816667,0.85,NaN,NaN,NaN,35.775417,139.814651
4,5,78151350,https://suumo.jp/ms/chuko/tokyo/sc_bunkyo/nc_7...,1800.0,26.29,1977,5.0,5.0,東京都文京区小石川５,茗荷谷駅,...,0.6500,NaN,NaN,0.775000,0.85,0.8000,NaN,NaN,35.718056,139.739929


In [25]:
# ── 2. Compute station-level aggregates ───────────────────────────────────────
data['price_per_sqm_man_yen'] = data['price_man_yen'] / data['area_sqm']

station_proxy = data.groupby('nearest_station').agg(
    station_avg_price_man_yen=('price_man_yen', 'mean'),
    station_avg_price_per_sqm=('price_per_sqm_man_yen', 'mean'),
    station_listing_count=('price_man_yen', 'count')
).reset_index()
station_proxy

,nearest_station,station_avg_price_man_yen,station_avg_price_per_sqm,station_listing_count
0,お台場海浜公園駅,17820.000000,220.892387,3
1,お花茶屋駅,3958.285714,63.189568,7
2,とうきょうスカイツリー駅,6364.500000,131.514335,4
3,ときわ台駅,6771.400000,91.459617,5
4,一之江駅,4966.500000,72.301080,4
...,...,...,...,...
387,鶯谷駅,7430.000000,151.706761,2
388,鶯谷駅駅,9880.000000,130.017108,1
389,鷺ノ宮駅,6920.000000,97.390952,3
390,麹町駅,19032.000000,221.298641,5


In [28]:
# ── 3. Merge back into listings(data) ───────────────────────────────────────────────
result = data.merge(station_proxy, on='nearest_station', how='left')
result = result.drop(columns='price_per_sqm_man_yen')
result.head()

,level_0,source_id,url,price_man_yen,area_sqm,year_built,floor_number,floors_total,address,nearest_station,...,condition_bathroom,condition_bedroom,condition_kitchen,condition_living_room,condition_toilet,latitude,longitude,station_avg_price_man_yen,station_avg_price_per_sqm,station_listing_count
0,1,20277732,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...,2390.0,52.89,1989,4.0,4.0,東京都足立区六月１,竹ノ塚駅,...,0.892500,0.85,0.8750,NaN,0.85,35.786911,139.800232,3328.666667,46.858457,9
1,2,20332918,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...,1580.0,29.16,1976,6.0,9.0,東京都足立区東和５,北綾瀬駅,...,0.950000,0.95,0.8875,NaN,0.85,35.775234,139.838226,3534.230769,58.652327,13
2,3,20344616,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...,2380.0,53.04,1974,3.0,8.0,東京都足立区梅田７-22-9,梅島駅,...,0.866667,0.93,0.9500,NaN,0.95,35.769596,139.801437,4031.333333,58.185682,6
3,4,78385243,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_7...,1700.0,40.10,1985,1.0,5.0,東京都足立区青井１-16-9,青井駅,...,0.816667,0.85,NaN,NaN,NaN,35.775417,139.814651,1700.000000,42.394015,1
4,5,78151350,https://suumo.jp/ms/chuko/tokyo/sc_bunkyo/nc_7...,1800.0,26.29,1977,5.0,5.0,東京都文京区小石川５,茗荷谷駅,...,0.775000,0.85,0.8000,NaN,NaN,35.718056,139.739929,12146.352941,194.428116,17


In [33]:
# ── 4. Save ───────────────────────────────────────────────────────────────────
result.to_csv('../data_dump/listings_with_avg_income.csv', index=False)
print(f"Done! {len(result)} rows, {len(result.columns)} columns")
print(result[['nearest_station', 'station_avg_price_man_yen', 'station_avg_price_per_sqm', 'station_listing_count']].drop_duplicates('nearest_station').head(10))


Done! 3303 rows, 34 columns
  nearest_station  station_avg_price_man_yen  station_avg_price_per_sqm  \
0            竹ノ塚駅                3328.666667                  46.858457   
1            北綾瀬駅                3534.230769                  58.652327   
2             梅島駅                4031.333333                  58.185682   
3             青井駅                1700.000000                  42.394015   
4            茗荷谷駅               12146.352941                 194.428116   
5            上板橋駅                4711.428571                  76.019435   
6             金町駅                6284.583333                  93.793410   
7           お花茶屋駅                3958.285714                  63.189568   
8             青砥駅                5302.571429                  78.245354   
9             田町駅               18779.018182                 261.201211   

   station_listing_count  
0                      9  
1                     13  
2                      6  
3                      1  
4          